In [9]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install mlflow[extras] hyperopt tensorflow scikit-learn pandas numpy

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 14.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   ---------------------------- ----------- 2.4/3.3 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 3.3/3.3 MB 10.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/14.1 MB ? eta -:--:--
   ------ --------------------------------- 2.4/14.1 MB 14.9 MB/s eta 0:00:01
   ------- -------------------------------- 2.6/14.1 MB 7.9 MB/s eta 0:00:02
   -------------------- ------------------- 7.1/14.1 MB 10.9 MB/s eta 0:00:01
   -------------------------- ------------- 9.2/14.1 MB 10.8 MB/s eta 0:00:01
   -------------------------------- ------- 11.3/14.1 MB 10.5 MB/s eta 0:00:01
   -------------------------------------- - 13.4/14.1 MB 10.4 MB/s et

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.40.64 which is incompatible.


In [14]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow import keras
import mlflow
from mlflow.models import infer_signature
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials


In [11]:
mlflow.set_tracking_uri("http://127.0.0.1:8080")

In [12]:
# Sets the current active experiment to the "Apple_Models" experiment and
# returns the Experiment metadata
apple_experiment = mlflow.set_experiment("wie_quality")

# Define a run name for this iteration of training.
# If this is not set, a unique name will be auto-generated for your run.
run_name = "wine_rf_test_1_1"

# Define an artifact path that the model will be saved to.
artifact_path = "rf_wine"


In [15]:
    # 1. Lectura de Datos

# Load the wine quality dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

# Create train/validation/test splits
train, test = train_test_split(data, test_size=0.25, random_state=42)
train_x = train.drop(["quality"], axis=1).values
train_y = train[["quality"]].values.ravel()
test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()

# Further split training data for validation
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)

# Create model signature for deployment
signature = infer_signature(train_x, train_y)



In [16]:
def create_and_train_model(learning_rate, momentum, epochs=10):
    """
    Create and train a neural network with specified hyperparameters.

    Returns:
        dict: Training results including model and metrics
    """
    # Normalize input features for better training stability
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)

    # Define model architecture
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(0.2),  # Add regularization
            keras.layers.Dense(32, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    # Compile with specified hyperparameters
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    # Train with early stopping for efficiency
    early_stopping = keras.callbacks.EarlyStopping(
        patience=3, restore_best_weights=True
    )

    # Train the model
    history = model.fit(
        train_x,
        train_y,
        validation_data=(valid_x, valid_y),
        epochs=epochs,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=0,  # Reduce output for cleaner logs
    )

    # Evaluate on validation set
    val_loss, val_rmse = model.evaluate(valid_x, valid_y, verbose=0)

    return {
        "model": model,
        "val_rmse": val_rmse,
        "val_loss": val_loss,
        "history": history,
        "epochs_trained": len(history.history["loss"]),
    }

In [17]:
def objective(params):
    """
    Objective function for hyperparameter optimization.
    This function will be called by Hyperopt for each trial.
    """
    with mlflow.start_run(nested=True):
        # Log hyperparameters being tested
        mlflow.log_params(
            {
                "learning_rate": params["learning_rate"],
                "momentum": params["momentum"],
                "optimizer": "SGD",
                "architecture": "64-32-1",
            }
        )

        # Train model with current hyperparameters
        result = create_and_train_model(
            learning_rate=params["learning_rate"],
            momentum=params["momentum"],
            epochs=15,
        )

        # Log training results
        mlflow.log_metrics(
            {
                "val_rmse": result["val_rmse"],
                "val_loss": result["val_loss"],
                "epochs_trained": result["epochs_trained"],
            }
        )

        # Log the trained model
        mlflow.tensorflow.log_model(result["model"], name="model", signature=signature)

        # Log training curves as artifacts
        import matplotlib.pyplot as plt

        plt.figure(figsize=(12, 4))

        plt.subplot(1, 2, 1)
        plt.plot(result["history"].history["loss"], label="Training Loss")
        plt.plot(result["history"].history["val_loss"], label="Validation Loss")
        plt.title("Model Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(
            result["history"].history["root_mean_squared_error"], label="Training RMSE"
        )
        plt.plot(
            result["history"].history["val_root_mean_squared_error"],
            label="Validation RMSE",
        )
        plt.title("Model RMSE")
        plt.xlabel("Epoch")
        plt.ylabel("RMSE")
        plt.legend()

        plt.tight_layout()
        plt.savefig("training_curves.png")
        mlflow.log_artifact("training_curves.png")
        plt.close()

        # Return loss for Hyperopt (it minimizes)
        return {"loss": result["val_rmse"], "status": STATUS_OK}


# Define search space for hyperparameters
search_space = {
    "learning_rate": hp.loguniform("learning_rate", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 0.9),
}

print("Search space defined:")
print("- Learning rate: 1e-5 to 1e-1 (log-uniform)")
print("- Momentum: 0.0 to 0.9 (uniform)")

Search space defined:
- Learning rate: 1e-5 to 1e-1 (log-uniform)
- Momentum: 0.0 to 0.9 (uniform)


In [18]:
# Create or set experiment
experiment_name = "wine-quality-optimization"
mlflow.set_experiment(experiment_name)

print(f"Starting hyperparameter optimization experiment: {experiment_name}")
print("This will run 15 trials to find optimal hyperparameters...")

with mlflow.start_run(run_name="hyperparameter-sweep"):
    # Log experiment metadata
    mlflow.log_params(
        {
            "optimization_method": "Tree-structured Parzen Estimator (TPE)",
            "max_evaluations": 15,
            "objective_metric": "validation_rmse",
            "dataset": "wine-quality",
            "model_type": "neural_network",
        }
    )

    # Run optimization
    trials = Trials()
    best_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=15,
        trials=trials,
        verbose=True,
    )

    # Find and log best results
    best_trial = min(trials.results, key=lambda x: x["loss"])
    best_rmse = best_trial["loss"]

    # Log optimization results
    mlflow.log_params(
        {
            "best_learning_rate": best_params["learning_rate"],
            "best_momentum": best_params["momentum"],
        }
    )
    mlflow.log_metrics(
        {
            "best_val_rmse": best_rmse,
            "total_trials": len(trials.trials),
            "optimization_completed": 1,
        }
    )

2025/11/01 19:54:45 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality-optimization' does not exist. Creating a new experiment.


Starting hyperparameter optimization experiment: wine-quality-optimization
This will run 15 trials to find optimal hyperparameters...
🏃 View run magnificent-roo-239 at: http://127.0.0.1:8080/#/experiments/170812052163285550/runs/a5fc320e20e34d729cd2643b6ef1f636

🧪 View experiment at: http://127.0.0.1:8080/#/experiments/170812052163285550

🏃 View run caring-turtle-418 at: http://127.0.0.1:8080/#/experiments/170812052163285550/runs/6a692cc5c0ac40f792cbb4205eb00022

🧪 View experiment at: http://127.0.0.1:8080/#/experiments/170812052163285550   

🏃 View run caring-ox-401 at: http://127.0.0.1:8080/#/experiments/170812052163285550/runs/c87d7c600ee542c0b0089798e6cde4b5

🧪 View experiment at: http://127.0.0.1:8080/#/experiments/170812052163285550   

🏃 View run placid-stag-415 at: http://127.0.0.1:8080/#/experiments/170812052163285550/runs/96f8bbb48c55427e94ffc2d0ae0822b5

🧪 View experiment at: http://127.0.0.1:8080/#/experiments/170812052163285550   

🏃 View run auspicious-eel-423 at: http://

In [19]:
# Create or set experiment
experiment_name = "wine-quality-optimization"
mlflow.set_experiment(experiment_name)

print(f"Starting hyperparameter optimization experiment: {experiment_name}")
print("This will run 15 trials to find optimal hyperparameters...")

with mlflow.start_run(run_name="hyperparameter-sweep"):
    # Log experiment metadata
    mlflow.log_params(
        {
            "optimization_method": "Tree-structured Parzen Estimator (TPE)",
            "max_evaluations": 15,
            "objective_metric": "validation_rmse",
            "dataset": "wine-quality",
            "model_type": "neural_network",
        }
    )

    # Run optimization
    trials = Trials()
    best_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=15,
        trials=trials,
        verbose=True,
    )

    # Find and log best results
    best_trial = min(trials.results, key=lambda x: x["loss"])
    best_rmse = best_trial["loss"]

    # Log optimization results
    mlflow.log_params(
        {
            "best_learning_rate": best_params["learning_rate"],
            "best_momentum": best_params["momentum"],
        }
    )
    mlflow.log_metrics(
        {
            "best_val_rmse": best_rmse,
            "total_trials": len(trials.trials),
            "optimization_completed": 1,
        }
    )

Starting hyperparameter optimization experiment: wine-quality-optimization
This will run 15 trials to find optimal hyperparameters...
🏃 View run valuable-quail-490 at: http://127.0.0.1:8080/#/experiments/170812052163285550/runs/893bbdcbb73c4a14a553f101f2043840

🧪 View experiment at: http://127.0.0.1:8080/#/experiments/170812052163285550

🏃 View run skillful-dove-231 at: http://127.0.0.1:8080/#/experiments/170812052163285550/runs/62989b225d7e4e779d979be4cc3cd0a5

🧪 View experiment at: http://127.0.0.1:8080/#/experiments/170812052163285550   

🏃 View run fun-cow-961 at: http://127.0.0.1:8080/#/experiments/170812052163285550/runs/415ab01acd2f46dc9eee47c3fd67d8bc

🧪 View experiment at: http://127.0.0.1:8080/#/experiments/170812052163285550   

🏃 View run defiant-bat-929 at: http://127.0.0.1:8080/#/experiments/170812052163285550/runs/118966cc03434ce7b65328a9df8bca2a

🧪 View experiment at: http://127.0.0.1:8080/#/experiments/170812052163285550   

🏃 View run unequaled-steed-438 at: http://12